# FASE 2 - Limpieza de Datos

1. Cargar dataset

In [1]:
import pandas as pd
from pathlib import Path

# Ruta del dataset desde el notebook
ruta_dataset = Path('../Data/2020-Apr-L.csv')
df_info = pd.read_csv(ruta_dataset)
# Ver primeras filas
df_info.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-04-01 00:00:00+00:00,view,1201465,2232732101407408685,apparel.shoes.slipons,samsung,230.38,568984877,e2456cef-2d4f-42b9-a53a-8893cb0c6851
1,2020-04-01 00:00:03+00:00,view,9500109,2232732104175649385,apparel.scarf,defender,25.05,530206135,e3c1fb4b-0a7e-457d-a0cf-5d1479e9aafc
2,2020-04-01 00:00:05+00:00,view,1201295,2232732101407408685,apparel.shoes.slipons,apple,489.05,635165586,48d05455-e287-4c44-84f9-76621e02b612
3,2020-04-01 00:00:09+00:00,view,9200640,2232732104343421549,apparel.scarf,defender,20.19,533896443,6a220235-f4d6-4987-a51e-8f315b3027fc
4,2020-04-01 00:00:11+00:00,view,17301057,2232732098446229999,apparel.shoes.sandals,unknown,31.18,543165860,7eaf3210-1d5e-4abc-8437-ae54960c3570


2. Resumir campos importantes del dataset

In [2]:
usuarios_unicos = df_info['user_id'].nunique()
productos_unicos = df_info['product_id'].nunique()
sesiones_unicas = df_info['user_session'].nunique()
total_interacciones = len(df_info)

rango_temporal = f"{df_info['event_time'].min()} a {df_info['event_time'].max()}"
promedio_interacciones_usuario = total_interacciones / usuarios_unicos

tabla_resumen = pd.DataFrame({
    "Métrica": [
        "Usuarios únicos",
        "Productos únicos",
        "Sesiones únicas",
        "Total interacciones",
        "Rango temporal cubierto",
        "Promedio de interacciones por usuario"
    ],
    "Valor": [
        usuarios_unicos,
        productos_unicos,
        sesiones_unicas,
        total_interacciones,
        rango_temporal,
        round(promedio_interacciones_usuario, 2)
    ]
})

tabla_resumen

,Métrica,Valor
0,Usuarios únicos,1379296
1,Productos únicos,60525
2,Sesiones únicas,2422079
3,Total interacciones,9601928
4,Rango temporal cubierto,2020-04-01 00:00:00+00:00 a 2020-04-30 23:59:5...
5,Promedio de interacciones por usuario,6.96


3. Porcentaje de tipo de evento

In [3]:
eventos = df_info['event_type'].value_counts().reset_index()
eventos.columns = ['event_type', 'cantidad']
eventos['porcentaje'] = round((eventos['cantidad'] / total_interacciones) * 100, 2)

eventos

,event_type,cantidad,porcentaje
0,view,9180901,95.62
1,cart,326641,3.40
2,purchase,94386,0.98


## Paso 1: Filtrado por categoría de moda

Solo vamos a filtrar los codigos de categoria "apparel", "accessories" y "sport.trainter" pertenecientes al dataset

In [4]:
categorias_validas = (
    df_info['category_code'].str.startswith('apparel.', na=False) |
    df_info['category_code'].str.startswith('accessories.', na=False) |
    df_info['category_code'].eq('sport.trainer')
)

df_moda = df_info.loc[categorias_validas].copy()

print("Registros antes del filtro:", len(df_info))
print("Registros después del filtro:", len(df_moda))
print("Registros eliminados:", len(df_info) - len(df_moda))


Registros antes del filtro: 9601928
Registros después del filtro: 9601928
Registros eliminados: 0


Visualizar resumen de registros de la categoria moda del dataset

In [5]:
categorias_despues = (
    df_moda['category_code']
    .value_counts()
    .reset_index()
)

categorias_despues.columns = ['category_code', 'cantidad']

categorias_despues

,category_code,cantidad
0,apparel.shoes,1789449
1,apparel.shoes.slipons,922814
2,sport.trainer,885813
3,apparel.shoes.keds,717024
4,accessories.bag,621076
5,apparel.shorts,604557
6,apparel.shoes.sandals,596802
7,apparel.shirt,538396
8,apparel.scarf,508156
9,apparel.underwear,492767


In [6]:
df_moda.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-04-01 00:00:00+00:00,view,1201465,2232732101407408685,apparel.shoes.slipons,samsung,230.38,568984877,e2456cef-2d4f-42b9-a53a-8893cb0c6851
1,2020-04-01 00:00:03+00:00,view,9500109,2232732104175649385,apparel.scarf,defender,25.05,530206135,e3c1fb4b-0a7e-457d-a0cf-5d1479e9aafc
2,2020-04-01 00:00:05+00:00,view,1201295,2232732101407408685,apparel.shoes.slipons,apple,489.05,635165586,48d05455-e287-4c44-84f9-76621e02b612
3,2020-04-01 00:00:09+00:00,view,9200640,2232732104343421549,apparel.scarf,defender,20.19,533896443,6a220235-f4d6-4987-a51e-8f315b3027fc
4,2020-04-01 00:00:11+00:00,view,17301057,2232732098446229999,apparel.shoes.sandals,unknown,31.18,543165860,7eaf3210-1d5e-4abc-8437-ae54960c3570


## Paso 2: Filtrado de duplicados exactos

Contar duplicados

In [7]:
hash_filas = pd.util.hash_pandas_object(
    df_moda,
    index=False
)
duplicados_antes = hash_filas.duplicated(
    keep='first'
).sum()

print("Duplicados exactos encontrados:", duplicados_antes)

Duplicados exactos encontrados: 0


Muestre de registros duplicados, incluyendo original y copias


In [8]:
mascara_duplicados = hash_filas.duplicated(
    keep=False
)

ejemplo_duplicados = df_moda.loc[
    mascara_duplicados
].head(10)

ejemplo_duplicados

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session


 Eliminacion de duplicados exactos

In [9]:
filas_antes = len(df_moda)

df_moda = df_moda.loc[
    ~hash_filas.duplicated(keep='first')
].copy()

filas_despues = len(df_moda)

print("Filas antes:", filas_antes)
print("Duplicados eliminados:", filas_antes - filas_despues)
print("Filas después:", filas_despues)

Filas antes: 9601928
Duplicados eliminados: 0
Filas después: 9601928


Recalcular hash sobre el dataframe ya limpio de duplicados

In [10]:
hash_filas = pd.util.hash_pandas_object(
    df_moda,
    index=False
)

duplicados_despues = hash_filas.duplicated(
    keep='first'
).sum()

print("Duplicados exactos después:", duplicados_despues)

Duplicados exactos después: 0


## Paso 3: Filtrado de valores nulos

Muestreo de campos con valores nulos

In [11]:

nulos = df_moda.isnull().sum().reset_index()

nulos.columns = [
    'columna',
    'cantidad_nulos'
]

nulos['porcentaje_nulos'] = (
    nulos['cantidad_nulos'] / len(df_moda) * 100
).round(2)

nulos

,columna,cantidad_nulos,porcentaje_nulos
0,event_time,0,0.0
1,event_type,0,0.0
2,product_id,0,0.0
3,category_id,0,0.0
4,category_code,0,0.0
5,brand,0,0.0
6,price,0,0.0
7,user_id,0,0.0
8,user_session,0,0.0


Ejemplos de campos con valores nulos

In [12]:
ejemplo_brand_nulo = df_moda[
    df_moda['brand'].isnull()
][
    [
        'event_time',
        'event_type',
        'product_id',
        'category_code',
        'brand',
        'price',
        'user_id',
        'user_session'
    ]
].head()

ejemplo_brand_nulo

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session


Tratamiento de valores nulos

In [13]:
filas_antes_nulos = len(df_moda)

# Reemplazar marcas faltantes por "unknown"
df_moda['brand'] = df_moda['brand'].fillna('unknown')

# Eliminar registros sin sesión
df_moda = df_moda.dropna(
    subset=['user_session']
).copy()

filas_despues_nulos = len(df_moda)

print("Filas antes:", filas_antes_nulos)
print("Filas eliminadas:", filas_antes_nulos - filas_despues_nulos)
print("Filas después:", filas_despues_nulos)

Filas antes: 9601928
Filas eliminadas: 0
Filas después: 9601928


Validacion de valores finales nulos

In [14]:
nulos_despues = df_moda.isnull().sum().reset_index()

nulos_despues.columns = [
    'columna',
    'cantidad_nulos'
]

nulos_despues

,columna,cantidad_nulos
0,event_time,0
1,event_type,0
2,product_id,0
3,category_id,0
4,category_code,0
5,brand,0
6,price,0
7,user_id,0
8,user_session,0


In [15]:
df_moda.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-04-01 00:00:00+00:00,view,1201465,2232732101407408685,apparel.shoes.slipons,samsung,230.38,568984877,e2456cef-2d4f-42b9-a53a-8893cb0c6851
1,2020-04-01 00:00:03+00:00,view,9500109,2232732104175649385,apparel.scarf,defender,25.05,530206135,e3c1fb4b-0a7e-457d-a0cf-5d1479e9aafc
2,2020-04-01 00:00:05+00:00,view,1201295,2232732101407408685,apparel.shoes.slipons,apple,489.05,635165586,48d05455-e287-4c44-84f9-76621e02b612
3,2020-04-01 00:00:09+00:00,view,9200640,2232732104343421549,apparel.scarf,defender,20.19,533896443,6a220235-f4d6-4987-a51e-8f315b3027fc
4,2020-04-01 00:00:11+00:00,view,17301057,2232732098446229999,apparel.shoes.sandals,unknown,31.18,543165860,7eaf3210-1d5e-4abc-8437-ae54960c3570


## Paso 4: Estandarizado de tiempo de evento

Se convierte el tipo de dato "event_time" a fecha y hora UTC

In [16]:
df_moda['event_time'] = pd.to_datetime(
    df_moda['event_time'],
    errors='coerce',
    utc=True
)

## Paso 5: Filtrado de precio menor o igual a 0

Se filtra las filas con el precio negativo o igual a cero

In [ ]:
precios_invalidos = df_moda[
    df_moda['price'] <= 0
][
    [
        'event_time',
        'event_type',
        'product_id',
        'category_code',
        'brand',
        'price',
        'user_id',
        'user_session'
    ]
]

print("Cantidad de registros con precio inválido:", len(precios_invalidos))

precios_invalidos.head()

Cantidad de registros con precio inválido: 0


,event_time,event_type,product_id,category_code,brand,price,user_id,user_session


In [18]:
filas_antes_precio = len(df_moda)

# ELIMINAR 0
df_moda = df_moda[
    df_moda['price'] > 0
].copy()

filas_despues_precio = len(df_moda)

print("Filas antes:", filas_antes_precio)
print("Registros con precio 0 eliminados:", filas_antes_precio - filas_despues_precio)
print("Filas después:", filas_despues_precio)

Filas antes: 9601928
Registros con precio 0 eliminados: 0
Filas después: 9601928


In [19]:
df_moda.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-04-01 00:00:00+00:00,view,1201465,2232732101407408685,apparel.shoes.slipons,samsung,230.38,568984877,e2456cef-2d4f-42b9-a53a-8893cb0c6851
1,2020-04-01 00:00:03+00:00,view,9500109,2232732104175649385,apparel.scarf,defender,25.05,530206135,e3c1fb4b-0a7e-457d-a0cf-5d1479e9aafc
2,2020-04-01 00:00:05+00:00,view,1201295,2232732101407408685,apparel.shoes.slipons,apple,489.05,635165586,48d05455-e287-4c44-84f9-76621e02b612
3,2020-04-01 00:00:09+00:00,view,9200640,2232732104343421549,apparel.scarf,defender,20.19,533896443,6a220235-f4d6-4987-a51e-8f315b3027fc
4,2020-04-01 00:00:11+00:00,view,17301057,2232732098446229999,apparel.shoes.sandals,unknown,31.18,543165860,7eaf3210-1d5e-4abc-8437-ae54960c3570


In [20]:
print("Registros finales:", len(df_moda))

Registros finales: 9601928


## Resumen despues de la limpieza de datos

In [21]:
print("Registros finales:", len(df_moda))
print("Nulos totales:", df_moda.isnull().sum().sum())
print("Precios <= 0:", (df_moda['price'] <= 0).sum())

hash_final = pd.util.hash_pandas_object(
    df_moda,
    index=False
)

print(
    "Duplicados exactos:",
    hash_final.duplicated(keep='first').sum()
)

print("\nTipos de evento:")
print(df_moda['event_type'].value_counts())

print("\nTipo de event_time:")
print(df_moda['event_time'].dtype)

Registros finales: 9601928
Nulos totales: 0
Precios <= 0: 0
Duplicados exactos: 0

Tipos de evento:
event_type
view        9180901
cart         326641
purchase      94386
Name: count, dtype: int64

Tipo de event_time:
datetime64[ns, UTC]


## Guardar Tabla

In [22]:
from pathlib import Path

ruta_salida = Path('../data/2020-Apr-L.csv')

df_moda.to_csv(
    ruta_salida,
    index=False
)

print("Dataset limpio guardado en:", ruta_salida)
print("Registros guardados:", len(df_moda))

Dataset limpio guardado en: ..\data\2020-Apr-L.csv
Registros guardados: 9601928


In [23]:
usuarios_unicos = df_moda['user_id'].nunique()
productos_unicos = df_moda['product_id'].nunique()
sesiones_unicas = df_moda['user_session'].nunique()
total_interacciones = len(df_moda)

rango_temporal = f"{df_moda['event_time'].min()} a {df_moda['event_time'].max()}"
promedio_interacciones_usuario = total_interacciones / usuarios_unicos

tabla_resumen = pd.DataFrame({
    "Métrica": [
        "Usuarios únicos",
        "Productos únicos",
        "Sesiones únicas",
        "Total interacciones",
        "Rango temporal cubierto",
        "Promedio de interacciones por usuario"
    ],
    "Valor": [
        usuarios_unicos,
        productos_unicos,
        sesiones_unicas,
        total_interacciones,
        rango_temporal,
        round(promedio_interacciones_usuario, 2)
    ]
})

tabla_resumen

,Métrica,Valor
0,Usuarios únicos,1379296
1,Productos únicos,60525
2,Sesiones únicas,2422079
3,Total interacciones,9601928
4,Rango temporal cubierto,2020-04-01 00:00:00+00:00 a 2020-04-30 23:59:5...
5,Promedio de interacciones por usuario,6.96
